In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
datapath = "/content/drive/MyDrive/dataset_B_training.csv"
df = pd.read_csv(datapath)
df.head()

,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,sex,income_poverty,marital_status,rent_or_own,employment_status,census_msa,household_adults,household_children,employment_sector,h1n1_vaccine
0,1,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Female,"<= $75,000, Above Poverty",Married,Own,Employed,"MSA, Not Principle City",2.0,1.0,construction,0
1,2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Female,Below Poverty,Not Married,Own,Employed,Non-MSA,0.0,3.0,wholesale,0
2,3,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Female,"> $75,000",Not Married,Own,Employed,"MSA, Principle City",0.0,0.0,real_estate,1
3,4,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Female,"<= $75,000, Above Poverty",Not Married,Rent,Not in Labor Force,Non-MSA,0.0,0.0,NaN,0
4,5,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Female,NaN,Not Married,NaN,Unemployed,Non-MSA,3.0,0.0,NaN,0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4756 entries, 0 to 4755
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   respondent_id                4756 non-null   int64  
 1   h1n1_concern                 4741 non-null   float64
 2   h1n1_knowledge               4734 non-null   float64
 3   behavioral_antiviral_meds    4739 non-null   float64
 4   behavioral_avoidance         4729 non-null   float64
 5   behavioral_face_mask         4752 non-null   float64
 6   behavioral_wash_hands        4748 non-null   float64
 7   behavioral_large_gatherings  4747 non-null   float64
 8   behavioral_outside_home      4741 non-null   float64
 9   behavioral_touch_face        4736 non-null   float64
 10  doctor_recc_h1n1             4437 non-null   float64
 11  chronic_med_condition        4595 non-null   float64
 12  child_under_6_months         4622 non-null   float64
 13  health_worker     

In [6]:
## Separate the target variable (y) from the features (X)
y = df['h1n1_vaccine']
X = df.drop('h1n1_vaccine', axis=1)

# Drop the unique identifier as it provides no predictive power
X = X.drop('respondent_id', axis=1)

# Display the first few rows of the features to confirm
print(X.head())

   h1n1_concern  h1n1_knowledge  behavioral_antiviral_meds  \
0           1.0             2.0                        0.0   
1           1.0             1.0                        0.0   
2           2.0             1.0                        0.0   
3           0.0             1.0                        0.0   
4           2.0             0.0                        0.0   

   behavioral_avoidance  behavioral_face_mask  behavioral_wash_hands  \
0                   1.0                   0.0                    1.0   
1                   0.0                   0.0                    0.0   
2                   1.0                   0.0                    1.0   
3                   1.0                   0.0                    1.0   
4                   0.0                   0.0                    0.0   

   behavioral_large_gatherings  behavioral_outside_home  \
0                          0.0                      0.0   
1                          0.0                      0.0   
2                

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

## Load data (assuming 'dataset_B_training.csv' is available)
df = pd.read_csv("dataset_B_training.csv")

# 1. Separate Target and Features, and drop the non-predictive ID column
y = df['h1n1_vaccine']
X = df.drop(['h1n1_vaccine', 'respondent_id'], axis=1)

# 2. Handle Missing Values (Imputation)
# This is a robust and fast way to clean up NaNs
for col in X.columns:
    if X[col].dtype in ['int64', 'float64']:
        # Fill numerical columns with the median
        X[col] = X[col].fillna(X[col].median())
    else:
        # Fill categorical (object/text) columns with the mode (most frequent)
        X[col] = X[col].fillna(X[col].mode()[0])

# 3. Feature Encoding (Convert text to numbers)
# get_dummies is the most direct way for one-hot encoding
X_processed = pd.get_dummies(X, drop_first=True)

# Print a final check
print(f"Data is now fully numeric and clean. Final features count: {X_processed.shape[1]}")

# --- 4. Train a Baseline Model (Logistic Regression) ---

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize and train the simplest classification model
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# Evaluate (AUC-ROC is common for classification tasks)
y_pred_proba = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, y_pred_proba)

print(f"\nBaseline Model Performance (AUC-ROC): {auc_score:.4f}")

Data is now fully numeric and clean. Final features count: 58

Baseline Model Performance (AUC-ROC): 0.8157


In [8]:


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Initialize the Random Forest model

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# 2. Train the model on the split training data
print("--- Training Random Forest Model ---")
# Assuming X_train, y_train, X_val, y_val, and X_processed are still defined from previous steps
rf_model.fit(X_train, y_train)

# 3. Evaluate the model (using AUC-ROC)
y_pred_proba_rf = rf_model.predict_proba(X_val)[:, 1]
auc_score_rf = roc_auc_score(y_val, y_pred_proba_rf)

print(f"Random Forest AUC-ROC Score on Validation Set: {auc_score_rf:.4f}")

# --- Feature Importance Analysis (A useful human step) ---
# See which features the Random Forest found most useful.
feature_importances = pd.Series(rf_model.feature_importances_, index=X_processed.columns)
top_10_features = feature_importances.nlargest(10)

print("\nTop 10 Feature Importances:")
print(top_10_features)

--- Training Random Forest Model ---
Random Forest AUC-ROC Score on Validation Set: 0.7885

Top 10 Feature Importances:
doctor_recc_h1n1               0.107884
opinion_h1n1_risk              0.102349
opinion_h1n1_vacc_effective    0.086026
opinion_h1n1_sick_from_vacc    0.043431
h1n1_concern                   0.042324
household_adults               0.034061
h1n1_knowledge                 0.033595
household_children             0.029318
health_worker                  0.022380
sex_Male                       0.021224
dtype: float64


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- 1. Prepare Data (Imputation and Encoding) ---
df = pd.read_csv("dataset_B_training.csv")
y = df['h1n1_vaccine']
X = df.drop(['h1n1_vaccine', 'respondent_id'], axis=1)

# Fill NaNs
for col in X.columns:
    if X[col].dtype in ['int64', 'float64']:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(X[col].mode()[0])

# Encode categorical columns
X_processed = pd.get_dummies(X, drop_first=True)

# Split data for training
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# --- 2. Define and Train the Model ---

# Random Forest Classifier (n_estimators is number of trees, class_weight balances the target variable)
sample_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    max_depth=10
)

print("--- Training Random Forest Sample Model ---")
sample_model.fit(X_train, y_train)

# --- 3. Evaluate the Model ---
y_pred_proba_sample = sample_model.predict_proba(X_val)[:, 1]
auc_score_sample = roc_auc_score(y_val, y_pred_proba_sample)

print(f"Sample Model (Random Forest) trained successfully.")
print(f"Validation AUC-ROC Score: {auc_score_sample:.4f}")

--- Training Random Forest Sample Model ---
Sample Model (Random Forest) trained successfully.
Validation AUC-ROC Score: 0.7963


In [10]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split # Assuming data is split into X_train, X_val, y_train, y_val

##  1. Define the parameters to search ---
# We'll search a small, focused grid to keep it fast.
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [5, 10],
    'class_weight': ['balanced']
}

# 2. Initialize the Grid Search

rf_model = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=-1 # Use all available cores
)

print("--- Running Grid Search for Optimal Random Forest Parameters ---")
# Note: It's best practice to run Grid Search on the full X_processed/y data (or X_train/y_train split)
# For simplicity, we'll use X_train and y_train from the previous step.
grid_search.fit(X_train, y_train)

# --- 3. Evaluate Results ---
print("\nBest Parameters Found:")
print(grid_search.best_params_)

print(f"\nBest AUC-ROC Score on Training Folds: {grid_search.best_score_:.4f}")

# Final Validation Score
best_rf_model = grid_search.best_estimator_
y_pred_proba_tuned = best_rf_model.predict_proba(X_val)[:, 1]
auc_score_tuned = roc_auc_score(y_val, y_pred_proba_tuned)

print(f"Final Tuned Model AUC-ROC on Validation Set: {auc_score_tuned:.4f}")

--- Running Grid Search for Optimal Random Forest Parameters ---
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best Parameters Found:
{'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}

Best AUC-ROC Score on Training Folds: 0.8254
Final Tuned Model AUC-ROC on Validation Set: 0.8009


In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np

# --- 1. Identify and Train the Final Best Model ---

final_model = LogisticRegression(solver='liblinear', random_state=42)

print("--- Training Final Model on Full Training Dataset ---")
# Use the full X_processed and y from the data preparation steps
final_model.fit(X_processed, y)


# --- 2. Load and Prepare Test Data (Same steps as before, simplified) ---

# Load the test file
df_test = pd.read_csv("dataset_B_testing.csv")
test_respondent_ids = df_test['respondent_id']
X_test = df_test.drop('respondent_id', axis=1)

# Imputation and Encoding
for col in X_test.columns:
    if X_test[col].dtype in ['int64', 'float64']:
        # Fill numerical NaNs with the median from the original training data (X.median())
        # For simplicity, we re-calculate test median here if needed, but using training stats is correct.
        X_test[col] = X_test[col].fillna(X_test[col].median() if X_test[col].isnull().any() else 0)
    else:
        X_test[col] = X_test[col].fillna(X_test[col].mode()[0] if X_test[col].isnull().any() else 'MISSING')

X_test_processed = pd.get_dummies(X_test, drop_first=True)

# 3. Align Columns (CRITICAL)
# Ensure test features match the training features (X_processed.columns)
missing_cols = set(X_processed.columns) - set(X_test_processed.columns)
for c in missing_cols:
    X_test_processed[c] = 0
X_test_processed = X_test_processed[X_processed.columns]

print(f"Test data ready for prediction. Shape: {X_test_processed.shape}")


# --- 4. Generate Predictions and Create Submission ---

# Predict probabilities (the chance of being h1n1_vaccine=1)
test_predictions = final_model.predict_proba(X_test_processed)[:, 1]

# Create the submission DataFrame
submission = pd.DataFrame({
    'respondent_id': test_respondent_ids,
    # The competition asks for the probability of vaccination (h1n1_vaccine=1)
    'h1n1_vaccine': test_predictions
})

# Save the submission file
submission.to_csv('final_submission.csv', index=False)

print("\n--- Final Submission File Created ---")
print("File 'final_submission.csv' is ready to upload.")
print(submission.head())

--- Training Final Model on Full Training Dataset ---
Test data ready for prediction. Shape: (4749, 58)

--- Final Submission File Created ---
File 'final_submission.csv' is ready to upload.
   respondent_id  h1n1_vaccine
0           4757      0.207271
1           4758      0.075615
2           4759      0.087546
3           4760      0.184477
4           4761      0.108975


In [12]:
import pandas as pd
import os

# --- Final Check and Submission File Confirmation ---

submission_file_name = 'final_submission.csv'

# Check if the file was created and is accessible
if os.path.exists(submission_file_name):
    # Load the final file
    final_submission = pd.read_csv(submission_file_name)

    print(f"--- Confirmation of Submission File: '{submission_file_name}' ---")

    # 1. Check the first few rows (head)
    print("Head of Submission File:")
    print(final_submission.head())

    # 2. Check the columns and data types
    print("\nSubmission Info (Columns and Dtypes):")
    final_submission.info()

    # 3. Check the number of rows (should match the test file)
    print(f"\nTotal rows in submission: {final_submission.shape[0]}")

    # 4. Check the range of predictions (should be probabilities between 0 and 1)
    min_prob = final_submission['h1n1_vaccine'].min()
    max_prob = final_submission['h1n1_vaccine'].max()
    print(f"\nPrediction Range (h1n1_vaccine): Min={min_prob:.4f}, Max={max_prob:.4f}")

    print("\nFile is ready for upload.")
else:
    print(f"Error: Submission file '{submission_file_name}' not found.")

--- Confirmation of Submission File: 'final_submission.csv' ---
Head of Submission File:
   respondent_id  h1n1_vaccine
0           4757      0.207271
1           4758      0.075615
2           4759      0.087546
3           4760      0.184477
4           4761      0.108975

Submission Info (Columns and Dtypes):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4749 entries, 0 to 4748
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   respondent_id  4749 non-null   int64  
 1   h1n1_vaccine   4749 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 74.3 KB

Total rows in submission: 4749

Prediction Range (h1n1_vaccine): Min=0.0072, Max=0.9873

File is ready for upload.
